# DATASET CURATION - MASKED ROI PROJECT


**Objectives**:

To create the following groups:
1. **Positive group**: BIRADS 0 that became BIRADS 3, 4, 5, 6 in the subsequent diagnostic study
2. **Negative group**: BIRADS 1, 2 and BIRADS 0 that became BIRADS 1, 2 in the subsequent diagnostic study


## 1. Prep

In [67]:
import pandas as pd
import numpy as np
import random
from tqdm import tqdm
import os

from IPython.display import display

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 500)

In [68]:
def get_stats(df, suffix=None):
    """Provides a quick summary of a dataframe."""
    try:
        print(f"DF shape: {df.shape}")
        print(f"# Patients: {df.empi_anon.nunique()}")
        print(f"# Cases: {df.acc_anon.nunique()}\n")
        print(f"# Images: {df.anon_dicom_path.nunique()}\n")
    except Exception as e:
        print(e)

In [69]:
## Read clinical and metadata rows
DATA_ROOT = "/mnt/cv_data/shared_datasets/EMBED/tables"

magview_path = os.path.join(DATA_ROOT, "EMBED_OpenData_clinical.csv")
metadata_path = os.path.join(DATA_ROOT, "EMBED_OpenData_metadata.csv")

metadata_full = pd.read_csv(metadata_path, dtype=str)
magview_full = pd.read_csv(magview_path, dtype=str)

In [70]:
# Selecting the following columns
meta_cols = [
    "empi_anon",
    "acc_anon",
    "ImageLateralityFinal",
    "ViewPosition",
    "study_date_anon",
    "FinalImageType",
    "anon_dicom_path",
    # "png_path",
    "StudyDescription",
    "ProtocolName",
    # "match_level",
    "num_ROI",
    "ROI_coords",
    "BreastImplantPresent",
]

mag_cols = [
    "empi_anon",
    "acc_anon",
    "tissueden", # newly added column for breast density by Meng
    "study_date_anon",
    "desc",
    "side",
    "asses",
    "path_severity",
    "bside",
    "procdate_anon",
    "pdate_anon",
]

In [71]:
metadata = metadata_full[meta_cols].copy()
magview = magview_full[mag_cols].copy()

In [72]:
metadata.study_date_anon = pd.to_datetime(metadata.study_date_anon, errors="coerce")
magview.study_date_anon = pd.to_datetime(magview.study_date_anon, errors="coerce")

In [73]:
# follow_up_period = metadata.groupby('empi_anon')['study_date_anon'].agg(['min', 'max'])
# follow_up_period['duration_years'] = (follow_up_period['max'] - follow_up_period['min']).dt.days / 365.25
# patients_with_5_years  = follow_up_period[follow_up_period['duration_years'] >= 4 ]

# Step 5: Get the patient IDs (empi_anon) that meet the 5-year criteria
# valid_patients = patients_with_5_years.index

# metadata  = metadata[metadata['empi_anon'].isin(valid_patients)].reset_index(drop=True)
# get_stats(metadata)

In [74]:
metadata["num_ROI"] = metadata["num_ROI"].fillna(0).astype(int)

metadata.num_ROI = metadata.num_ROI.astype(int)

## 2. METADATA: 2D MLO & CC

In [75]:
# EMBED 2D (MLO and CC)
meta_2d = metadata.loc[
    (metadata.FinalImageType == "2D") & (metadata.ViewPosition.isin(["MLO", "CC"]))
]
get_stats(meta_2d)

DF shape: (337098, 12)
# Patients: 23252
# Cases: 72652

# Images: 337098



In [76]:
# def get_image_stats(df):
#     """Provides a quick summary of the number of unique images and the ROIs."""
#     temp_df = pd.merge(df, meta_2d, on=["empi_anon", "acc_anon"], how="left")
#     temp_df = temp_df.loc[(temp_df.side == temp_df.ImageLateralityFinal)]
#     temp_df.drop_duplicates(subset="png_path", inplace=True)
#     print(f"# PNG PATH: {int(temp_df.png_path.nunique())}")
#     print(f"# ROI: {int(temp_df.num_ROI.sum())}")
#     print(f"{temp_df.num_ROI.value_counts()}")
#     del temp_df

def get_image_stats(df):
    """Provides a quick summary of the number of unique images and the ROIs."""
    temp_df = pd.merge(df, meta_2d, on=["empi_anon", "acc_anon"], how="left")
    temp_df = temp_df.loc[(temp_df.side == temp_df.ImageLateralityFinal)]
    temp_df.drop_duplicates(subset="anon_dicom_path", inplace=True)
    print(f"# DICOM PATH: {int(temp_df.anon_dicom_path.nunique())}")
    print(f"# ROI: {int(temp_df.num_ROI.sum())}")
    print(f"{temp_df.num_ROI.value_counts()}")
    del temp_df


## 3. Screening

In [77]:
# SCREENING
screening_magview = magview.loc[magview.desc.str.contains("screen", case=False)].copy()
get_stats(screening_magview)

DF shape: (58888, 11)
# Patients: 20460
# Cases: 55956

'DataFrame' object has no attribute 'anon_dicom_path'


### 3.1. Creating entries for the negative contralateral breast in bilateral examinations

```
MAGVIEW only has entries if a finding exists.

This means that if an exam is a bilateral exam and only one of the breast has a finding, the contralateral breast (negative) won't have an entry.

This would be problematic at the time when we need to merge with METADATA, because the contralateral breast would be excluded.

Therefore, we would need to create rows for the negative contralateral breast.
```

In [78]:
def get_exam_laterality(row):
    """A convenient function to get the exam laterality to be used with DF.apply() instead of iterating over each row."""
    if "bilat" in row.desc.lower():
        return "B"
    elif "left" in row.desc.lower():
        return "L"
    elif "right" in row.desc.lower():
        return "R"
    else:
        return None

In [79]:
# Applying the get_exam_laterality function
screening_magview["exam_laterality"] = screening_magview.apply(
    get_exam_laterality, axis=1
)

In [80]:
screening_magview.exam_laterality.value_counts(dropna=False)

exam_laterality
B    56558
L     1180
R     1150
Name: count, dtype: int64

In [81]:
screening_magview.side.value_counts(dropna=False)

side
NaN    39661
L       8264
R       8082
B       2881
Name: count, dtype: int64

In [82]:
# side == nan --> B
screening_magview.side = screening_magview.side.fillna("B")

In [83]:
# create copy for assigning B to R
screening_magview_r = screening_magview.loc[screening_magview.side == "B"].copy()
screening_magview_r.side = screening_magview.side.str.replace("B", "R")

# assigning B to L
screening_magview.side = screening_magview.side.str.replace("B", "L")

# appending R and L
screening_magview = pd.concat([screening_magview, screening_magview_r])

In [84]:
print(screening_magview.side.value_counts(dropna=False))
print(screening_magview.shape)

side
L    50806
R    50624
Name: count, dtype: int64
(101430, 12)


In [85]:
screening_magview = screening_magview.sort_values(
    ["empi_anon", "acc_anon", "study_date_anon"]
).drop_duplicates()
screening_magview

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality
31489,10000879,6992096043050201,3.0,2018-02-16,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
31489,10000879,6992096043050201,3.0,2018-02-16,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
10198,10009146,4190527469809995,4.0,2014-07-04,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
10198,10009146,4190527469809995,4.0,2014-07-04,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
20474,10015693,1334581155737139,2.0,2015-10-11,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,A,NaN,NaN,NaN,NaN,B
...,...,...,...,...,...,...,...,...,...,...,...,...
21119,99996622,9655172659462321,3.0,2016-06-04,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
25708,99999564,4369225803558884,3.0,2017-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
25708,99999564,4369225803558884,3.0,2017-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
31493,99999564,8832872399780580,3.0,2019-02-27,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B


In [86]:
exam_lat_b = screening_magview.loc[screening_magview.exam_laterality == "B"]
exam_lat_b.sample(2)

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality
55867,99171944,4267970797059565,3.0,2015-08-23,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
75705,96784972,3157046492247500,2.0,2019-09-18,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B


In [87]:
# We want to aggregate all the sides for each bilateral exam so that we can filter those having only a single side.
exam_lat_b_agg = exam_lat_b.groupby("acc_anon")["side"].apply("".join).reset_index()
exam_lat_b_agg.sample(2)

,acc_anon,side
5884,1988623592166709,R
32819,6473148645580785,LR


In [88]:
exam_lat_b_agg.side.value_counts()

side
LR        42769
L          4938
R          4882
RL          615
LL          123
RR          114
LLR          75
LRR          43
RLR          25
LLRR         20
RRL          16
RLL          12
LLL          11
RLLR          5
LRL           5
RRR           4
LRLR          4
RRLL          3
RRLR          2
LLLRRR        2
LRRR          1
RRRR          1
LLLLLR        1
LLRRR         1
LLLRR         1
LLLR          1
RRRLLL        1
LRRL          1
Name: count, dtype: int64

In [89]:
exam_lat_b_side_r = exam_lat_b_agg.loc[~(exam_lat_b_agg.side.str.contains("L"))].copy()
exam_lat_b_side_l = exam_lat_b_agg.loc[~(exam_lat_b_agg.side.str.contains("R"))].copy()

In [90]:
screening_magview_right_to_left = (
    screening_magview.loc[screening_magview.acc_anon.isin(exam_lat_b_side_r.acc_anon)]
    .copy()
    .drop_duplicates()
)
screening_magview_left_to_right = (
    screening_magview.loc[screening_magview.acc_anon.isin(exam_lat_b_side_l.acc_anon)]
    .copy()
    .drop_duplicates()
)

In [91]:
# Creating the negative Left side
screening_magview_right_to_left.loc[
    screening_magview_right_to_left.side == "R", "side"
] = "L"
screening_magview_right_to_left.loc[
    screening_magview_right_to_left.side == "L", "asses"
] = "N"
screening_magview_right_to_left.loc[
    screening_magview_right_to_left.side == "L", "path_severity"
] = np.nan

screening_magview_right_to_left

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality
37325,10033806,1069386741434572,3.0,2019-10-05,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
64534,10043985,1960584382049532,2.0,2018-04-18,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
59088,10043985,3613575521057039,2.0,2017-03-01,MG Screening Bilateral,L,N,NaN,NaN,NaN,NaN,B
42847,10043985,9492972692582499,2.0,2014-05-14,MG Screening Bilateral,L,N,NaN,NaN,NaN,NaN,B
8607,10065082,6346759651734606,2.0,2015-03-03,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
...,...,...,...,...,...,...,...,...,...,...,...,...
11436,99853035,2905584160156737,3.0,2015-02-15,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
33776,99853035,6677454260490853,3.0,2019-02-07,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
67642,99860105,6470240272862407,2.0,2018-03-19,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
12820,99871644,5176060292067455,3.0,2015-06-26,MG Screening Bilateral,L,N,NaN,NaN,NaN,NaN,B


In [92]:
# Creating the negative Right side
screening_magview_left_to_right.loc[
    screening_magview_left_to_right.side == "L", "side"
] = "R"
screening_magview_left_to_right.loc[
    screening_magview_left_to_right.side == "R", "asses"
] = "N"
screening_magview_left_to_right.loc[
    screening_magview_left_to_right.side == "R", "path_severity"
] = np.nan

screening_magview_left_to_right

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality
20474,10015693,1334581155737139,2.0,2015-10-11,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
25070,10023113,5135241747022662,2.0,2016-10-05,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
63407,10029585,3189592535497441,3.0,2017-06-11,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
62320,10042753,1955284757719450,2.0,2017-06-15,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
37004,10044241,3993319361430024,2.0,2019-07-27,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
...,...,...,...,...,...,...,...,...,...,...,...,...
80934,99881569,1140879824262422,2.0,2021-01-05,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
43901,99881569,3921887412575009,2.0,2013-11-02,MG Screening Bilateral w/CAD,R,N,NaN,NaN,NaN,NaN,B
9588,99908618,9288525074493489,1.0,2014-08-14,MG Screening Bilateral w/CAD,R,N,NaN,L,2014-08-26,2014-08-27 00:00:00,B
6519,99957941,2224428804635608,2.0,2014-06-03,MG Screening Bilateral,R,N,NaN,NaN,NaN,NaN,B


In [93]:
# Merging the original and the two negative contralaterals
screening_magview_with_contralat = (
    pd.concat(
        [
            screening_magview,
            screening_magview_left_to_right,
            screening_magview_right_to_left,
        ]
    )
    .sort_values(["empi_anon", "acc_anon", "study_date_anon"])
    .drop_duplicates()
)
screening_magview_with_contralat.sample(2)

OUTPUT_ROOT = "/mnt/cv_data/users/mengxu/Longitudinal_Mammogram_Alignment/output_csv"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

screening_magview_with_contralat.to_csv(
    os.path.join(OUTPUT_ROOT, "EMBED_OpenData_magview_with_controlateral.csv"), index=False
)

In [94]:
get_stats(meta_2d)

DF shape: (337098, 12)
# Patients: 23252
# Cases: 72652

# Images: 337098



In [95]:
get_stats(screening_magview_with_contralat)
display(screening_magview_with_contralat)

DF shape: (110396, 12)
# Patients: 20460
# Cases: 55956

'DataFrame' object has no attribute 'anon_dicom_path'


,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality
31489,10000879,6992096043050201,3.0,2018-02-16,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
31489,10000879,6992096043050201,3.0,2018-02-16,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
10198,10009146,4190527469809995,4.0,2014-07-04,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
10198,10009146,4190527469809995,4.0,2014-07-04,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
20474,10015693,1334581155737139,2.0,2015-10-11,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,A,NaN,NaN,NaN,NaN,B
...,...,...,...,...,...,...,...,...,...,...,...,...
21119,99996622,9655172659462321,3.0,2016-06-04,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
25708,99999564,4369225803558884,3.0,2017-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B
25708,99999564,4369225803558884,3.0,2017-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B
31493,99999564,8832872399780580,3.0,2019-02-27,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B


In [96]:
get_image_stats(screening_magview_with_contralat)

# DICOM PATH: 266406
# ROI: 8984
num_ROI
0.0    258162
1.0      7618
2.0       532
3.0        76
4.0        16
5.0         2
Name: count, dtype: int64


### 3.2. BIRADS 0

In [97]:
b0 = screening_magview_with_contralat.loc[
    screening_magview_with_contralat.asses.isin(["A"])
]

get_stats(b0)
get_image_stats(b0)

DF shape: (10876, 12)
# Patients: 7747
# Cases: 8829

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 25445
# ROI: 8758
num_ROI
0.0    17392
1.0     7455
2.0      511
3.0       69
4.0       16
5.0        2
Name: count, dtype: int64


### 3.3. BIRADS 1, 2

In [98]:
b12 = screening_magview_with_contralat.loc[
    screening_magview_with_contralat.asses.isin(["B", "N"])
]

get_stats(b12)
get_image_stats(b12)

DF shape: (99482, 12)


# Patients: 19665
# Cases: 54081

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 241479
# ROI: 392
num_ROI
0.0    241142
1.0       292
2.0        35
3.0        10
Name: count, dtype: int64


## 4. Diagnostic

In [99]:
diag_magview = magview.loc[magview.desc.str.contains("diag", case=False)]

get_stats(diag_magview)
print()
print(f"Asses Counts:\n{diag_magview.asses.value_counts()}")

DF shape: (22888, 11)
# Patients: 9656
# Cases: 16814

'DataFrame' object has no attribute 'anon_dicom_path'

Asses Counts:
asses
B    8794
P    5563
N    4193
S    3063
A     580
K     386
M     284
X      25
Name: count, dtype: int64


## 5. Screening BIRADS 0 and Diagnostic

In [100]:
b0_dx = pd.merge(b0, diag_magview, on="empi_anon", suffixes=[None, "_dx"])
b0_dx = b0_dx.loc[
    (b0_dx.side == b0_dx.side_dx) | (b0_dx.side_dx == "B") | (b0_dx.side_dx.isna())
]

In [101]:
# Getting only subsequent diagnostic studies within 3 months
b0_dx["delta_date_dx"] = (b0_dx.study_date_anon_dx - b0_dx.study_date_anon).dt.days
b0_dx_3mo = b0_dx.loc[b0_dx.delta_date_dx.isin(range(0, 91))]
b0_dx_3mo.sample(1)

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality,acc_anon_dx,tissueden_dx,study_date_anon_dx,desc_dx,side_dx,asses_dx,path_severity_dx,bside_dx,procdate_anon_dx,pdate_anon_dx,delta_date_dx
11728,61231761,2575872677730952,1.0,2012-12-07,MG Screening Bilateral w/CAD,L,A,NaN,NaN,NaN,NaN,B,6647476089634189,1.0,2012-12-23,MG Diagnostic Left,L,P,NaN,NaN,NaN,NaN,16


### 5.1. BIRADS 0 (Screening) --> BIRADS 1, 2 (Diagnostic)

In [102]:
b0_12dx = b0_dx_3mo.loc[b0_dx_3mo.asses_dx.isin(["N", "B"])].copy()
get_stats(b0_12dx)
get_image_stats(b0_12dx)

DF shape: (3755, 23)
# Patients: 2924
# Cases: 3169

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 8069
# ROI: 2719
num_ROI
0.0    5583
1.0    2285
2.0     178
3.0      16
4.0       5
5.0       2
Name: count, dtype: int64


### 5.2. BIRADS 0 (Screening) --> BIRADS 3, 4, 5, 6 (Diagnostic)

In [103]:
b0_3456dx = magview.loc[
    magview.asses.isin(["K"]) | magview.path_severity.isin([0, 1])
].copy()

get_stats(b0_3456dx)
get_image_stats(b0_3456dx)
display(b0_3456dx)

DF shape: (395, 11)
# Patients: 216
# Cases: 313

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 806
# ROI: 41
num_ROI
0.0    779
1.0     23
3.0      2
4.0      1
8.0      1
Name: count, dtype: int64


,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon
248,41737961,3103295389430403,3.0,2013-01-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN
249,41737961,3103295389430403,3.0,2013-01-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN
787,82818555,9133832708678084,3.0,2013-11-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN
1151,12628486,8298312705312234,1.0,2013-09-05,MG Diagnostic Right,R,K,0.0,R,2013-10-10,2013-10-10 00:00:00
1372,75428728,3608864100750486,3.0,2014-01-30,MG Diagnostic Left,L,K,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
81451,87877516,2183052495313832,2.0,2021-01-09,MG Diagnostic Bilateral w/Tomo/CAD,L,K,0.0,L,2021-02-15,2021-02-22 00:00:00
81452,87877516,2183052495313832,2.0,2021-01-09,MG Diagnostic Bilateral w/Tomo/CAD,L,K,0.0,L,2021-02-15,2021-02-22 00:00:00
81453,87877516,2183052495313832,2.0,2021-01-09,MG Diagnostic Bilateral w/Tomo/CAD,L,K,0.0,L,2021-02-15,2021-02-22 00:00:00
81558,99703607,1965937622916299,3.0,2019-12-08,MG Diagnostic Right w/CAD,R,K,0.0,R,2019-12-29,2020-01-01 00:00:00


## 6. Negative group

In [104]:
# Negative group = BIRADS_12 + BIRADS_0_12dx
neg_group = pd.concat([b12, b0_12dx])
neg_group.drop_duplicates(inplace=True)

get_stats(neg_group)
get_image_stats(neg_group)

DF shape: (103023, 23)
# Patients: 19877
# Cases: 54639

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 249372
# ROI: 3066
num_ROI
0.0    246588
1.0      2544
2.0       207
3.0        26
4.0         5
5.0         2
Name: count, dtype: int64


In [105]:
# Include only ones with negative follow-up after 1 year
neg_group_b12 = pd.merge(neg_group, b12, on=["empi_anon"], suffixes=(None, "_1yrfu"))

neg_group_b12 = neg_group_b12.loc[(neg_group_b12.side == neg_group_b12.side_1yrfu)]

neg_group_b12["delta_date_1yrfu"] = (
    neg_group_b12.study_date_anon_1yrfu - neg_group_b12.study_date_anon
).dt.days

get_stats(neg_group_b12)
get_image_stats(neg_group_b12)

neg_group_b12.sample(2)

DF shape: (419008, 35)
# Patients: 19665
# Cases: 54427

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 247192
# ROI: 2211
num_ROI
0.0    245151
1.0      1891
2.0       133
3.0        14
4.0         3
Name: count, dtype: int64


,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality,acc_anon_dx,tissueden_dx,study_date_anon_dx,desc_dx,side_dx,asses_dx,path_severity_dx,bside_dx,procdate_anon_dx,pdate_anon_dx,delta_date_dx,acc_anon_1yrfu,tissueden_1yrfu,study_date_anon_1yrfu,desc_1yrfu,side_1yrfu,asses_1yrfu,path_severity_1yrfu,bside_1yrfu,procdate_anon_1yrfu,pdate_anon_1yrfu,exam_laterality_1yrfu,delta_date_1yrfu
779869,96949385,1523229917180665,1.0,2018-01-04,MG Screening Bilateral w/CAD,R,B,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9295974798158165,1.0,2013-12-29,MG Screening Bilateral w/CAD,R,B,NaN,NaN,NaN,NaN,B,-1467
365903,50967173,7345322863627695,2.0,2018-12-20,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2968057007324103,2.0,2016-12-16,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,-734


In [106]:
neg_group_1yrfu = neg_group_b12.loc[(neg_group_b12.delta_date_1yrfu > 360)]
get_stats(neg_group_1yrfu)
get_image_stats(neg_group_1yrfu)

DF shape: (158091, 35)
# Patients: 11590
# Cases: 34180

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 154037
# ROI: 1491
num_ROI
0.0    152666
1.0      1265
2.0        94
3.0        10
4.0         2
Name: count, dtype: int64


In [107]:
neg_group_1yrfu_first_study = neg_group_1yrfu.sort_values(
    ["empi_anon", "acc_anon", "study_date_anon_1yrfu"]
).drop_duplicates(subset=["acc_anon", "side"])  # to only get the first followup study
get_stats(neg_group_1yrfu_first_study)
get_image_stats(neg_group_1yrfu_first_study)

DF shape: (63835, 35)
# Patients: 11590
# Cases: 34180

'DataFrame' object has no attribute 'anon_dicom_path'
# DICOM PATH: 154037
# ROI: 1491
num_ROI
0.0    152666
1.0      1265
2.0        94
3.0        10
4.0         2
Name: count, dtype: int64


In [108]:
neg_group_1yrfu_first_study.path_severity.value_counts()

path_severity
4.0    25
2.0     7
0.0     4
3.0     1
Name: count, dtype: int64

In [109]:
# Exclude any patient with any biopsy result
neg_group_1yrfu_first_study_no_biopsy = neg_group_1yrfu_first_study.loc[
    neg_group_1yrfu_first_study.path_severity.isna()
].copy()

In [110]:
# Merging with METADATA to get the images
neg_group_1yrfu_first_study_no_biopsy_images = pd.merge(
    neg_group_1yrfu_first_study_no_biopsy,
    meta_2d,
    on=["empi_anon", "acc_anon", "study_date_anon"],
)
neg_group_1yrfu_first_study_no_biopsy_images = (
    neg_group_1yrfu_first_study_no_biopsy_images.loc[
        (
            neg_group_1yrfu_first_study_no_biopsy_images.side
            == neg_group_1yrfu_first_study_no_biopsy_images.ImageLateralityFinal
        )
    ]
)
# neg_group_1yrfu_first_study_no_biopsy_images.drop_duplicates(
#     subset="png_path", inplace=True
# )

neg_group_1yrfu_first_study_no_biopsy_images.drop_duplicates(
    subset="anon_dicom_path", inplace=True
)

get_stats(neg_group_1yrfu_first_study_no_biopsy_images)

DF shape: (152877, 44)
# Patients: 11552
# Cases: 33925

# Images: 152877



In [111]:
print(f"ROIs = {neg_group_1yrfu_first_study_no_biopsy_images.num_ROI.sum()}")
print(neg_group_1yrfu_first_study_no_biopsy_images.num_ROI.value_counts())

ROIs = 1438
num_ROI
0    151544
1      1240
2        83
3         8
4         2
Name: count, dtype: int64


## 7. Positive Group

In [112]:
pos_group_images = pd.merge(
    b0_3456dx, meta_2d, on=["empi_anon", "acc_anon", "study_date_anon"]
)
pos_group_images = pos_group_images.loc[
    (pos_group_images.side == pos_group_images.ImageLateralityFinal)
]
pos_group_images.drop_duplicates(subset="anon_dicom_path", inplace=True)
get_stats(pos_group_images)
display(pos_group_images)

DF shape: (806, 20)
# Patients: 209
# Cases: 303

# Images: 806



,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,ImageLateralityFinal,ViewPosition,FinalImageType,anon_dicom_path,StudyDescription,ProtocolName,num_ROI,ROI_coords,BreastImplantPresent
1,41737961,3103295389430403,3.0,2013-01-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN,R,CC,2D,images/cohort_1/41737961/1.2.846.113971.3.58.1...,MG Diagnostic Mammo Bilateral,RCC,0,[],NO
4,41737961,3103295389430403,3.0,2013-01-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN,R,MLO,2D,images/cohort_1/41737961/1.2.846.113971.3.58.1...,MG Diagnostic Mammo Bilateral,RMLO,0,[],NO
5,41737961,3103295389430403,3.0,2013-01-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN,R,MLO,2D,images/cohort_1/41737961/1.2.846.113971.3.58.1...,MG Diagnostic Mammo Bilateral,RMLO,0,[],NO
12,82818555,9133832708678084,3.0,2013-11-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN,R,CC,2D,images/cohort_1/82818555/1.2.848.113974.3.61.1...,MG Diagnostic Mammo Bilateral,RCC,0,[],NO
13,82818555,9133832708678084,3.0,2013-11-10,MG Diagnostic Mammo Bilateral,R,K,NaN,NaN,NaN,NaN,R,CC,2D,images/cohort_1/82818555/1.2.848.113974.3.61.1...,MG Diagnostic Mammo Bilateral,RMCC,0,[],NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1492,87877516,2183052495313832,2.0,2021-01-09,MG Diagnostic Bilateral w/Tomo/CAD,L,K,0.0,L,2021-02-15,2021-02-22 00:00:00,L,MLO,2D,images/cohort_2/87877516/1.2.842.113975.3.62.1...,MG Diagnostic Bilateral w/Tomo/CAD,L MLO ComboHD,0,[],NO
1509,99703607,1965937622916299,3.0,2019-12-08,MG Diagnostic Right w/CAD,R,K,0.0,R,2019-12-29,2020-01-01 00:00:00,R,CC,2D,images/cohort_2/99703607/1.2.844.113978.3.59.1...,MG Diagnostic Right w/CAD,R CC,0,[],NO
1510,99703607,1965937622916299,3.0,2019-12-08,MG Diagnostic Right w/CAD,R,K,0.0,R,2019-12-29,2020-01-01 00:00:00,R,MLO,2D,images/cohort_2/99703607/1.2.844.113978.3.59.1...,MG Diagnostic Right w/CAD,R MLO,0,[],NO
1511,85936969,4786838232916669,3.0,2019-09-10,MG Diagnostic Right w/CAD,R,K,0.0,R,2019-10-11,2019-10-15 00:00:00,R,CC,2D,images/cohort_2/85936969/1.2.848.113975.3.60.1...,MG Diagnostic Right w/CAD,3D_ROUTINE+2D_ROUTINE,0,[],NO


In [113]:
print(f"ROIs  = {pos_group_images.num_ROI.sum()}")
print(pos_group_images.num_ROI.value_counts())

ROIs  = 41
num_ROI
0    779
1     23
3      2
4      1
8      1
Name: count, dtype: int64


## 8. Excluding Images from the Negative Group that are found in the Positive Group using acc_anon and side

In [114]:
# Merge negatives and positive groups
neg_pos = pd.merge(
    neg_group_1yrfu_first_study_no_biopsy_images,
    pos_group_images,
    on=["empi_anon", "acc_anon", "side"],
    suffixes=["_neg", "_pos"],
)

In [115]:
# Create new KeyID of acc_anon + side on negative group and negative+positive group
neg_pos["acc_anon_side"] = neg_pos.acc_anon + neg_pos.side

In [116]:
neg_group_1yrfu_first_study_no_biopsy_images["acc_anon_side"] = (
    neg_group_1yrfu_first_study_no_biopsy_images.acc_anon
    + neg_group_1yrfu_first_study_no_biopsy_images.side
)
neg_group_1yrfu_first_study_no_biopsy_images.sample(2)

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality,acc_anon_dx,tissueden_dx,study_date_anon_dx,desc_dx,side_dx,asses_dx,path_severity_dx,bside_dx,procdate_anon_dx,pdate_anon_dx,delta_date_dx,acc_anon_1yrfu,tissueden_1yrfu,study_date_anon_1yrfu,desc_1yrfu,side_1yrfu,asses_1yrfu,path_severity_1yrfu,bside_1yrfu,procdate_anon_1yrfu,pdate_anon_1yrfu,exam_laterality_1yrfu,delta_date_1yrfu,ImageLateralityFinal,ViewPosition,FinalImageType,anon_dicom_path,StudyDescription,ProtocolName,num_ROI,ROI_coords,BreastImplantPresent,acc_anon_side
64052,29322854,9206384487755661,3.0,2014-08-20,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3363306468143869,3.0,2015-10-14,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,420,L,CC,2D,images/cohort_2/29322854/1.2.841.113977.3.66.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L CC Combo,0,[],NO,9206384487755661L
132486,49785812,4005210579842847,1.0,2014-05-15,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9075588067466645,1.0,2016-05-18,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,734,R,CC,2D,images/cohort_1/49785812/1.2.841.113977.3.64.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R CC,0,[],NO,4005210579842847R


In [117]:
# Removing any images that are found in the positive group from the negative group using the created KeyID (acc_anon+side)
neg_group_final = neg_group_1yrfu_first_study_no_biopsy_images.loc[
    ~neg_group_1yrfu_first_study_no_biopsy_images.acc_anon_side.isin(
        neg_pos.acc_anon_side
    )
]
neg_group_final.sample(2)

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality,acc_anon_dx,tissueden_dx,study_date_anon_dx,desc_dx,side_dx,asses_dx,path_severity_dx,bside_dx,procdate_anon_dx,pdate_anon_dx,delta_date_dx,acc_anon_1yrfu,tissueden_1yrfu,study_date_anon_1yrfu,desc_1yrfu,side_1yrfu,asses_1yrfu,path_severity_1yrfu,bside_1yrfu,procdate_anon_1yrfu,pdate_anon_1yrfu,exam_laterality_1yrfu,delta_date_1yrfu,ImageLateralityFinal,ViewPosition,FinalImageType,anon_dicom_path,StudyDescription,ProtocolName,num_ROI,ROI_coords,BreastImplantPresent,acc_anon_side
62946,28929095,6979747475347598,3.0,2015-12-16,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8580819513989526,2.0,2016-12-15,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,365,R,CC,2D,images/cohort_1/28929095/1.2.846.113971.3.62.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R CC,0,[],YES,6979747475347598R
231011,78934105,1180394334120489,2.0,2017-12-22,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2897869921234959,2.0,2019-11-10,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,688,R,CC,2D,images/cohort_1/78934105/1.2.847.113979.3.60.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R CC ComboHD,0,[],NO,1180394334120489R


In [118]:
get_stats(neg_group_final)

print(f"ROIs  = {neg_group_final.num_ROI.sum()}")
print(neg_group_final.num_ROI.value_counts())

DF shape: (152877, 45)
# Patients: 11552
# Cases: 33925

# Images: 152877

ROIs  = 1438
num_ROI
0    151544
1      1240
2        83
3         8
4         2
Name: count, dtype: int64


## 9. Saving and Exporting

In [119]:
columns_to_save = [
    "empi_anon",
    "acc_anon",
    "tissueden",
    "anon_dicom_path",
    "desc",
    "ProtocolName",
    "asses",
    "path_severity",
    "study_date_anon",
    "side",
    "ImageLateralityFinal",
    "bside",
    "ViewPosition",
    # "match_level",
    "num_ROI",
    "ROI_coords",
]

### Final Filtering 
- Exclude patients from Postive group 
- Exclude patients with breast implants
- keep only patients of the negative group with enough 5 years follow up 
- Additional image-level filtering. Determine which rows to keep.

In [120]:
# Filter again to make sure there is no overlap with the positive group

# Exclude negative patients using the final patient set from POSITIVE_GROUP_FINAL.csv
positive_final_path = os.path.join(OUTPUT_ROOT, "POSITIVE_GROUP_FINAL.csv")
positive_final_df = pd.read_csv(positive_final_path)

# The positive final CSV uses patient_id, while the negative pipeline still uses empi_anon
positive_final_patient_ids = set(positive_final_df["patient_id"].dropna().astype(str).unique())
negative_patient_ids = set(neg_group_final["empi_anon"].dropna().astype(str).unique())

common_patients_final = negative_patient_ids & positive_final_patient_ids
print(f"Patients in POSITIVE_GROUP_FINAL.csv: {len(positive_final_patient_ids)}")
print(f"Patients in current negative group before final exclusion: {len(negative_patient_ids)}")
print(f"Overlapping patients to exclude from negative group: {len(common_patients_final)}")

neg_group_final_filtered = neg_group_final[
    ~neg_group_final["empi_anon"].astype(str).isin(positive_final_patient_ids)
].copy()

print("Negative group after exclusion using POSITIVE_GROUP_FINAL.csv patient set:")
get_stats(neg_group_final_filtered)

# Optional sanity check
remaining_overlap = set(neg_group_final_filtered["empi_anon"].dropna().astype(str).unique()) & positive_final_patient_ids
print(f"Remaining patient overlap after exclusion: {len(remaining_overlap)}")



Patients in POSITIVE_GROUP_FINAL.csv: 381
Patients in current negative group before final exclusion: 11552
Overlapping patients to exclude from negative group: 236
Negative group after exclusion using POSITIVE_GROUP_FINAL.csv patient set:
DF shape: (150343, 45)
# Patients: 11316
# Cases: 33268

# Images: 150343

Remaining patient overlap after exclusion: 0


In [121]:
# remove patients with breast implantat
# Find the patient IDs (empi_anon) with a breast implant
patients_with_implants = neg_group_final_filtered[
    neg_group_final_filtered["BreastImplantPresent"] == "YES"
]["empi_anon"].unique()

# Remove all rows for these patients
neg_group_final_filtered_final = neg_group_final_filtered[
    ~neg_group_final_filtered["empi_anon"].isin(patients_with_implants)
]
neg_group_final_filtered_final = neg_group_final_filtered_final[
    ~neg_group_final_filtered_final["ProtocolName"].str.contains(
        "SCC|SMLO|RMLOACIMF|RMLOAC|CCID|MLOID|MLOIMF|MCC|MLOAC|CCAC|MLOIDIMF|MLONP|CCNP|MLOAX|MLOAXIMF|TAN|CCRM|CCAX|CEDM|CESM|CCRL|MLOACNP|LMLOAC",
        case=False,
        na=False,
    )
    | neg_group_final_filtered_final["ProtocolName"].isna()
]
get_stats(neg_group_final_filtered_final)
display(neg_group_final_filtered_final)

DF shape: (136828, 45)
# Patients: 10914
# Cases: 32098

# Images: 136828



,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality,acc_anon_dx,tissueden_dx,study_date_anon_dx,desc_dx,side_dx,asses_dx,path_severity_dx,bside_dx,procdate_anon_dx,pdate_anon_dx,delta_date_dx,acc_anon_1yrfu,tissueden_1yrfu,study_date_anon_1yrfu,desc_1yrfu,side_1yrfu,asses_1yrfu,path_severity_1yrfu,bside_1yrfu,procdate_anon_1yrfu,pdate_anon_1yrfu,exam_laterality_1yrfu,delta_date_1yrfu,ImageLateralityFinal,ViewPosition,FinalImageType,anon_dicom_path,StudyDescription,ProtocolName,num_ROI,ROI_coords,BreastImplantPresent,acc_anon_side
0,10015693,1334581155737139,2.0,2015-10-11,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2281263876413228,2.0,2018-01-06,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,818,R,MLO,2D,images/cohort_1/10015693/1.2.847.113972.3.64.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R MLO ComboHD,0,[],NO,1334581155737139R
1,10015693,1334581155737139,2.0,2015-10-11,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2281263876413228,2.0,2018-01-06,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,818,R,CC,2D,images/cohort_1/10015693/1.2.847.113972.3.64.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R CC ComboHD,0,[],NO,1334581155737139R
5,10019048,6465041439526495,4.0,2013-03-11,MG Screening Bilateral,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8545300395264346,4.0,2016-07-24,MG Screening Bilateral,L,N,NaN,NaN,NaN,NaN,B,1231,L,CC,2D,images/cohort_2/10019048/1.2.849.113978.3.57.1...,MG Screening Bilateral,NaN,0,[],NaN,6465041439526495L
7,10019048,6465041439526495,4.0,2013-03-11,MG Screening Bilateral,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8545300395264346,4.0,2016-07-24,MG Screening Bilateral,L,N,NaN,NaN,NaN,NaN,B,1231,L,MLO,2D,images/cohort_2/10019048/1.2.849.113978.3.57.1...,MG Screening Bilateral,NaN,0,[],NaN,6465041439526495L
8,10019048,6465041439526495,4.0,2013-03-11,MG Screening Bilateral,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8545300395264346,4.0,2016-07-24,MG Screening Bilateral,R,N,NaN,NaN,NaN,NaN,B,1231,R,CC,2D,images/cohort_2/10019048/1.2.849.113978.3.57.1...,MG Screening Bilateral,NaN,0,[],NaN,6465041439526495R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302200,99996622,5582628875236699,3.0,2014-07-24,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,B,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9655172659462321,3.0,2016-06-04,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,681,R,CC,2D,images/cohort_1/99996622/1.2.843.113975.3.60.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R CC Combo,0,[],NO,5582628875236699R
302202,99999564,4369225803558884,3.0,2017-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8832872399780580,3.0,2019-02-27,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,673,L,CC,2D,images/cohort_1/99999564/1.2.845.113971.3.61.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L CC Combo,0,[],NO,4369225803558884L
302204,99999564,4369225803558884,3.0,2017-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8832872399780580,3.0,2019-02-27,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,673,L,MLO,2D,images/cohort_1/99999564/1.2.845.113971.3.61.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L MLO Combo,0,[],NO,4369225803558884L
302205,99999564,4369225803558884,3.0,2017-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8832872399780580,3.0,2019-02-27,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,673,R,CC,2D,images/cohort_1/9999

In [122]:
# keep only patients of the negative group with enough 5 years follow up
neg_group_final_filtered_final.study_date_anon = pd.to_datetime(
    neg_group_final_filtered_final.study_date_anon, errors="coerce"
)

follow_up_period = neg_group_final_filtered_final.groupby("empi_anon")[
    "study_date_anon"
].agg(["min", "max"])
follow_up_period["duration_years"] = (
    follow_up_period["max"].dt.year - follow_up_period["min"].dt.year
)
patients_with_5_years = follow_up_period[follow_up_period["duration_years"] >= 5]
valid_patients = patients_with_5_years.index

neg_group_final_5_years = neg_group_final_filtered_final[
    neg_group_final_filtered_final["empi_anon"].isin(valid_patients)
].reset_index(drop=True)
get_stats(neg_group_final_5_years)
display(neg_group_final_5_years)

DF shape: (56641, 45)
# Patients: 2418
# Cases: 13072

# Images: 56641



,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality,acc_anon_dx,tissueden_dx,study_date_anon_dx,desc_dx,side_dx,asses_dx,path_severity_dx,bside_dx,procdate_anon_dx,pdate_anon_dx,delta_date_dx,acc_anon_1yrfu,tissueden_1yrfu,study_date_anon_1yrfu,desc_1yrfu,side_1yrfu,asses_1yrfu,path_severity_1yrfu,bside_1yrfu,procdate_anon_1yrfu,pdate_anon_1yrfu,exam_laterality_1yrfu,delta_date_1yrfu,ImageLateralityFinal,ViewPosition,FinalImageType,anon_dicom_path,StudyDescription,ProtocolName,num_ROI,ROI_coords,BreastImplantPresent,acc_anon_side
0,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500827897014911,3.0,2014-07-02,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,376,L,CC,2D,images/cohort_2/10093833/1.2.846.113971.3.58.1...,MG Screening Bilateral w/CAD,L CC,0,[],NO,2030506250163251L
1,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500827897014911,3.0,2014-07-02,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,376,L,MLO,2D,images/cohort_2/10093833/1.2.846.113971.3.58.1...,MG Screening Bilateral w/CAD,L MLO,0,[],NO,2030506250163251L
2,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6488770689649000,3.0,2015-07-24,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,763,R,MLO,2D,images/cohort_2/10093833/1.2.846.113971.3.58.1...,MG Screening Bilateral w/CAD,R MLO,0,[],NO,2030506250163251R
3,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6488770689649000,3.0,2015-07-24,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,763,R,CC,2D,images/cohort_2/10093833/1.2.846.113971.3.58.1...,MG Screening Bilateral w/CAD,R CC,0,[],NO,2030506250163251R
4,10093833,2500827897014911,3.0,2014-07-02,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6488770689649000,3.0,2015-07-24,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,387,L,CC,2D,images/cohort_2/10093833/1.2.842.113974.3.60.1...,MG Screening Bilateral w/CAD,L CC,0,[],NO,2500827897014911L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56636,99986224,8107409307566891,1.0,2018-05-22,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8848125344172315,1.0,2019-08-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,460,R,CC,2D,images/cohort_1/99986224/1.2.843.113977.3.60.1...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R CC ComboHD,0,[],NO,8107409307566891R
56637,99986224,9061973132112039,1.0,2013-04-10,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1388973192449589,1.0,2015-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,745,L,MLO,2D,images/cohort_1/99986224/1.2.843.113972.3.57.1...,MG Screening Bilateral w/CAD,L MLO,0,[],NO,9061973132112039L
56638,99986224,9061973132112039,1.0,2013-04-10,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1388973192449589,1.0,2015-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,745,L,CC,2D,images/cohort_1/99986224/1.2.843.113972.3.57.1...,MG Screening Bilateral w/CAD,L CC,0,[],NO,9061973132112039L
56639,99986224,9061973132112039,1.0,2013-04-10,MG Screening Bilateral w/CAD,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1388973192449589,1.0,2015-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,745,R,CC,2D,images/cohort_1/99986224/1.2.843.113972.3.57.1...,MG Screening Bilateral w/CAD,R CC,0,[],NO,90

In [123]:
EMBED_ROOT = "/mnt/cv_data/shared_datasets/EMBED"

def to_abs_dcm_path(p: str) -> str:
    """
    Convert relative EMBED path to absolute path.
    Example:
        "images/cohort_1/.../xxx.dcm"
        ->
        "/mnt/cv_data/shared_datasets/EMBED/images/cohort_1/.../xxx.dcm"
    """
    if pd.isna(p):
        return None
    
    p = p.lstrip("/")  # remove leading slash if exists
    
    return os.path.join(EMBED_ROOT, p)


# def anon_dicom_path_fix(DICOMPathStr):
#     return DICOMPathStr.replace("/mnt/NAS2/mammo/anon_dicom", "/storage2/images")


result_df_neg_group_final_new_path = neg_group_final_5_years.copy()
result_df_neg_group_final_new_path["anon_dicom_path_local"] = (
    result_df_neg_group_final_new_path["anon_dicom_path"].apply(to_abs_dcm_path)
)
result_df_neg_group_final_new_path = result_df_neg_group_final_new_path.drop(
    columns=["anon_dicom_path"]
)
result_df_neg_group_final_new_path = result_df_neg_group_final_new_path.rename(
    columns={"anon_dicom_path_local": "anon_dicom_path"}
)

display(result_df_neg_group_final_new_path)

,empi_anon,acc_anon,tissueden,study_date_anon,desc,side,asses,path_severity,bside,procdate_anon,pdate_anon,exam_laterality,acc_anon_dx,tissueden_dx,study_date_anon_dx,desc_dx,side_dx,asses_dx,path_severity_dx,bside_dx,procdate_anon_dx,pdate_anon_dx,delta_date_dx,acc_anon_1yrfu,tissueden_1yrfu,study_date_anon_1yrfu,desc_1yrfu,side_1yrfu,asses_1yrfu,path_severity_1yrfu,bside_1yrfu,procdate_anon_1yrfu,pdate_anon_1yrfu,exam_laterality_1yrfu,delta_date_1yrfu,ImageLateralityFinal,ViewPosition,FinalImageType,StudyDescription,ProtocolName,num_ROI,ROI_coords,BreastImplantPresent,acc_anon_side,anon_dicom_path
0,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500827897014911,3.0,2014-07-02,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,376,L,CC,2D,MG Screening Bilateral w/CAD,L CC,0,[],NO,2030506250163251L,/mnt/cv_data/shared_datasets/EMBED/images/coho...
1,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500827897014911,3.0,2014-07-02,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,376,L,MLO,2D,MG Screening Bilateral w/CAD,L MLO,0,[],NO,2030506250163251L,/mnt/cv_data/shared_datasets/EMBED/images/coho...
2,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6488770689649000,3.0,2015-07-24,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,763,R,MLO,2D,MG Screening Bilateral w/CAD,R MLO,0,[],NO,2030506250163251R,/mnt/cv_data/shared_datasets/EMBED/images/coho...
3,10093833,2030506250163251,NaN,2013-06-21,MG Screening Bilateral w/CAD,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6488770689649000,3.0,2015-07-24,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,763,R,CC,2D,MG Screening Bilateral w/CAD,R CC,0,[],NO,2030506250163251R,/mnt/cv_data/shared_datasets/EMBED/images/coho...
4,10093833,2500827897014911,3.0,2014-07-02,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6488770689649000,3.0,2015-07-24,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,387,L,CC,2D,MG Screening Bilateral w/CAD,L CC,0,[],NO,2500827897014911L,/mnt/cv_data/shared_datasets/EMBED/images/coho...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56636,99986224,8107409307566891,1.0,2018-05-22,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8848125344172315,1.0,2019-08-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,460,R,CC,2D,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R CC ComboHD,0,[],NO,8107409307566891R,/mnt/cv_data/shared_datasets/EMBED/images/coho...
56637,99986224,9061973132112039,1.0,2013-04-10,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1388973192449589,1.0,2015-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,745,L,MLO,2D,MG Screening Bilateral w/CAD,L MLO,0,[],NO,9061973132112039L,/mnt/cv_data/shared_datasets/EMBED/images/coho...
56638,99986224,9061973132112039,1.0,2013-04-10,MG Screening Bilateral w/CAD,L,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1388973192449589,1.0,2015-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,L,N,NaN,NaN,NaN,NaN,B,745,L,CC,2D,MG Screening Bilateral w/CAD,L CC,0,[],NO,9061973132112039L,/mnt/cv_data/shared_datasets/EMBED/images/coho...
56639,99986224,9061973132112039,1.0,2013-04-10,MG Screening Bilateral w/CAD,R,N,NaN,NaN,NaN,NaN,B,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1388973192449589,1.0,2015-04-25,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R,N,NaN,NaN,NaN,NaN,B,745,R,CC,2D,MG Screening Bilateral w/CAD,R CC,0,[],NO,9061973132112039R,/mnt/cv_data/shared_datasets/EMBED

In [124]:
result_df_neg_group_final_new_path[columns_to_save].to_csv(
    os.path.join(OUTPUT_ROOT, "NEGATIVE_GROUP_FINAL_temp.csv"), index=False
)

In [125]:
import pandas as pd

neg_path = os.path.join(OUTPUT_ROOT, "NEGATIVE_GROUP_FINAL_temp.csv")

df_neg_group = pd.read_csv(neg_path)
get_stats(df_neg_group)

DF shape: (56641, 15)
# Patients: 2418
# Cases: 13072

# Images: 56641



In [126]:
# Step 1: Define a helper function to determine rows to keep
def filter_images(group):
    # Count protocol frequencies within the examination
    protocol_counts = group["ProtocolName"].value_counts()

    # Add protocol frequency column
    group["ProtocolFrequency"] = group["ProtocolName"].map(protocol_counts)

    # Sort by ViewPosition, ProtocolFrequency, and keep first occurrence if there's a tie
    group = group.sort_values(
        by=["ViewPosition", "ProtocolFrequency"], ascending=[True, False]
    )
    group = group.drop_duplicates(
        subset=["ImageLateralityFinal", "ViewPosition"], keep="last"
    )
    # Drop the helper column before returning
    group = group.drop(columns=["ProtocolFrequency"])
    return group


# Step 2: Apply the helper function group-wise
result_df_neg_group = df_neg_group.groupby(
    ["empi_anon", "acc_anon"], group_keys=False
).apply(filter_images)
# Step 3: Reset index if necessary
result_df_neg_group = result_df_neg_group.reset_index(drop=True)

/tmp/ipykernel_1271352/2179069200.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(filter_images)


In [127]:
get_stats(result_df_neg_group)
display(result_df_neg_group)

DF shape: (49690, 15)
# Patients: 2418
# Cases: 13072

# Images: 49690



,empi_anon,acc_anon,tissueden,anon_dicom_path,desc,ProtocolName,asses,path_severity,study_date_anon,side,ImageLateralityFinal,bside,ViewPosition,num_ROI,ROI_coords
0,10093833,2030506250163251,NaN,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,L CC,N,NaN,2013-06-21,L,L,NaN,CC,0,[]
1,10093833,2030506250163251,NaN,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,R CC,N,NaN,2013-06-21,R,R,NaN,CC,0,[]
2,10093833,2030506250163251,NaN,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,L MLO,N,NaN,2013-06-21,L,L,NaN,MLO,0,[]
3,10093833,2030506250163251,NaN,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,R MLO,N,NaN,2013-06-21,R,R,NaN,MLO,0,[]
4,10093833,2500827897014911,3.0,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,L CC,N,NaN,2014-07-02,L,L,NaN,CC,0,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49685,99986224,8107409307566891,1.0,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screen Bilat w/Tomo/CAD Stnd Protocol,R MLO ComboHD,N,NaN,2018-05-22,R,R,NaN,MLO,0,[]
49686,99986224,9061973132112039,1.0,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,L CC,N,NaN,2013-04-10,L,L,NaN,CC,0,[]
49687,99986224,9061973132112039,1.0,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,R CC,N,NaN,2013-04-10,R,R,NaN,CC,0,[]
49688,99986224,9061973132112039,1.0,/mnt/cv_data/shared_datasets/EMBED/images/coho...,MG Screening Bilateral w/CAD,L MLO,N,NaN,2013-04-10,L,L,NaN,MLO,0,[]


### Save the final CSV

In [128]:
result_df_neg_group = result_df_neg_group.rename(
    columns={
        "empi_anon": "patient_id",
        "acc_anon": "exam_id",
        # "ImageLateralityFinall": "laterality", # The original code has a typo here so it keeps the original column name 
        "ViewPosition": "view",
        "anon_dicom_path": "file_path_dcm",
        "tissueden": "density",
    }
)

In [129]:
result_df_neg_group.to_csv(os.path.join(OUTPUT_ROOT, "NEGATIVE_GROUP_FINAL.csv"), index=False)


# END

## The following code is to check, after including the density column, if a patient's density matches the original CSV

In [141]:

result_df_neg_group = pd.read_csv(os.path.join(OUTPUT_ROOT, "NEGATIVE_GROUP_FINAL.csv"))
magview_full = pd.read_csv(os.path.join(DATA_ROOT, "EMBED_OpenData_clinical.csv"), dtype=str)

# print the total number of non-null density values in the final negative group 
print("Non-null density values in result_df_neg_group:", result_df_neg_group["density"].notna().sum())
print("Total rows in result_df_neg_group:", len(result_df_neg_group))

Non-null density values in result_df_neg_group: 49566
Total rows in result_df_neg_group: 49690


In [ ]:
# test a random patient
patient_id = "10283168"

# clinical source
magview_sub = magview_full[
    magview_full["empi_anon"].astype(str) == patient_id
][["empi_anon", "acc_anon", "study_date_anon", "tissueden"]].copy()

# final negative CSV
result_sub = result_df_neg_group[
    result_df_neg_group["patient_id"].astype(str) == patient_id
][["patient_id", "exam_id", "density", "view", "file_path_dcm"]].copy()

print("=== magview (clinical source) ===")
display(magview_sub)

print("=== final result (CSV) ===")
display(result_sub)

# rename + unify types
magview_exam = magview_sub.rename(
    columns={
        "empi_anon": "patient_id",
        "acc_anon": "exam_id",
        "tissueden": "density_clinical"
    }
).copy()

magview_exam["patient_id"] = magview_exam["patient_id"].astype(str)
magview_exam["exam_id"] = magview_exam["exam_id"].astype(str)

result_exam = result_sub.rename(
    columns={"density": "density_final"}
).copy()

result_exam["patient_id"] = result_exam["patient_id"].astype(str)
result_exam["exam_id"] = result_exam["exam_id"].astype(str)

# one row per exam
magview_exam = magview_exam.drop_duplicates(subset=["patient_id", "exam_id"])
result_exam = result_exam.drop_duplicates(subset=["patient_id", "exam_id"])

# only compare exams that exist in final negative CSV
compare_df = pd.merge(
    result_exam,
    magview_exam[["patient_id", "exam_id", "study_date_anon", "density_clinical"]],
    on=["patient_id", "exam_id"],
    how="left"
)

compare_df["match"] = (
    compare_df["density_final"].astype(float)
    .fillna(-1)
    ==
    compare_df["density_clinical"].astype(float)
    .fillna(-1)
)

print("=== comparison for exams existing in final negative CSV ===")
display(compare_df)

print(compare_df[["patient_id", "exam_id", "density_final", "density_clinical", "match"]])

=== magview (clinical source) ===


,empi_anon,acc_anon,study_date_anon,tissueden
50050,10283168,7644593990180906,2014-01-08,3.0
55583,10283168,7760651270109819,2015-01-14,3.0
60938,10283168,4998372131887571,2016-01-14,3.0
66256,10283168,4696161302815255,2017-01-19,3.0
72216,10283168,2196804195539044,2018-02-08,3.0
78065,10283168,3864122305830679,2019-03-12,3.0
81278,10283168,6583335119427107,2020-06-30,2.0


=== final result (CSV) ===


,patient_id,exam_id,density,view,file_path_dcm
108,10283168,2196804195539044,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...
109,10283168,2196804195539044,3.0,MLO,/mnt/cv_data/shared_datasets/EMBED/images/coho...
110,10283168,3864122305830679,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...
111,10283168,3864122305830679,3.0,MLO,/mnt/cv_data/shared_datasets/EMBED/images/coho...
112,10283168,4696161302815255,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...
113,10283168,4696161302815255,3.0,MLO,/mnt/cv_data/shared_datasets/EMBED/images/coho...
114,10283168,4998372131887571,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...
115,10283168,4998372131887571,3.0,MLO,/mnt/cv_data/shared_datasets/EMBED/images/coho...
116,10283168,7644593990180906,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...
117,10283168,7644593990180906,3.0,MLO,/mnt/cv_data/shared_datasets/EMBED/images/coho...


=== comparison for exams existing in final negative CSV ===


,patient_id,exam_id,density_final,view,file_path_dcm,study_date_anon,density_clinical,match
0,10283168,2196804195539044,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...,2018-02-08,3.0,True
1,10283168,3864122305830679,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...,2019-03-12,3.0,True
2,10283168,4696161302815255,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...,2017-01-19,3.0,True
3,10283168,4998372131887571,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...,2016-01-14,3.0,True
4,10283168,7644593990180906,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...,2014-01-08,3.0,True
5,10283168,7760651270109819,3.0,CC,/mnt/cv_data/shared_datasets/EMBED/images/coho...,2015-01-14,3.0,True


  patient_id           exam_id  density_final density_clinical  match
0   10283168  2196804195539044            3.0              3.0   True
1   10283168  3864122305830679            3.0              3.0   True
2   10283168  4696161302815255            3.0              3.0   True
3   10283168  4998372131887571            3.0              3.0   True
4   10283168  7644593990180906            3.0              3.0   True
5   10283168  7760651270109819            3.0              3.0   True


In [144]:
# test all patients in the final negative CSV

# Build exam-level clinical source table from magview_full
magview_exam = magview_full[
    ["empi_anon", "acc_anon", "study_date_anon", "tissueden"]
].copy()

magview_exam = magview_exam.rename(
    columns={
        "empi_anon": "patient_id",
        "acc_anon": "exam_id",
        "tissueden": "density_clinical"
    }
)

# Keep the requested columns from final negative CSV
result_exam = result_df_neg_group[
    ["patient_id", "exam_id", "density", "view", "file_path_dcm"]
].copy()

result_exam = result_exam.rename(columns={"density": "density_final"})

# Make merge keys consistent
magview_exam["patient_id"] = magview_exam["patient_id"].astype(str)
magview_exam["exam_id"] = magview_exam["exam_id"].astype(str)

result_exam["patient_id"] = result_exam["patient_id"].astype(str)
result_exam["exam_id"] = result_exam["exam_id"].astype(str)

# Drop duplicate exam-level rows in magview_full
magview_exam = magview_exam.drop_duplicates(
    subset=["patient_id", "exam_id", "study_date_anon", "density_clinical"]
)

# Drop duplicate rows in final result for the same image-level row set if needed
result_exam = result_exam.drop_duplicates()

# Merge: use final negative CSV as the main table
compare_df = pd.merge(
    result_exam,
    magview_exam,
    on=["patient_id", "exam_id"],
    how="left"
)

# Compare densities safely
compare_df["density_final_num"] = pd.to_numeric(compare_df["density_final"], errors="coerce")
compare_df["density_clinical_num"] = pd.to_numeric(compare_df["density_clinical"], errors="coerce")

compare_df["match"] = (
    (compare_df["density_final_num"] == compare_df["density_clinical_num"])
    |
    (compare_df["density_final_num"].isna() & compare_df["density_clinical_num"].isna())
)

# Keep only mismatched rows
mismatch_df = compare_df[~compare_df["match"]].copy()

# Display only the requested source/final columns plus match info
mismatch_df = mismatch_df[
    [
        "patient_id",        # from result_df_neg_group
        "exam_id",           # from result_df_neg_group
        "density_final",     # result_df_neg_group density
        "view",
        "file_path_dcm",
        "study_date_anon",   # from magview_full
        "density_clinical",  # magview_full tissueden
        "match"
    ]
]

print(f"Total rows in result_df_neg_group: {len(result_exam)}")
print(f"Total mismatched rows: {len(mismatch_df)}")

display(mismatch_df)

Total rows in result_df_neg_group: 49690
Total mismatched rows: 0


,patient_id,exam_id,density_final,view,file_path_dcm,study_date_anon,density_clinical,match
